In [1]:
!pip install causal-conv1d>=1.4.0 --no-build-isolation

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [2]:
!pip install mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 5.1 MB/s eta 0:00:0000:01
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 45.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 25.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 66.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 47.2 MB/s eta 0:00:00
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.2.post1-cp312-cp312-linux_x86_64.whl size=322288410 sha256=7a67070c1e7e99c95abd1319623f044e8a1b3fb46f774bfdea949f0a4fc79638
  Stored in directory: /root/.cache/pip/wheels/da/67/03/99148d6eeaa4ec2855d71295ac83bcbc8ba7b41a2982992c63
Successfully built mamba-ssm


In [3]:
!pip install mamba-ssm[causal-conv1d] --no-build-isolation

In [4]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.1 MB/s eta 0:00:0000:01


In [5]:
import os
import ast
import wfdb
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

class PTBXLDataset(Dataset):
    def __init__(self, data_path, metadata_csv, scp_csv, sampling_rate=100, folds=None):
        super().__init__()
        self.data_path = data_path
        self.sampling_rate = sampling_rate
        
        # 1. Load metadata
        self.df = pd.read_csv(metadata_csv, index_col='ecg_id')
        self.df['scp_codes'] = self.df.scp_codes.apply(lambda x: ast.literal_eval(x))
        
        # --- CRITICAL FIX 1: Filter by Folds to prevent Data Leakage ---
        if folds is not None:
            self.df = self.df[self.df.strat_fold.isin(folds)]
            print(f"Filtered dataset to folds: {folds}. Total records to load: {len(self.df)}")
        
        self.scp_statements = pd.read_csv(scp_csv, index_col=0)
        self.diagnostic_df = self.scp_statements[self.scp_statements.diagnostic == 1]
        
        self.superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
        self.labels = self._get_labels()
        
        # 2. THE TRUE RAM CACHE
        self.signals = []
        
        print(f"Pre-loading {sampling_rate}Hz dataset into RAM. This will take a moment...")
        for idx in tqdm(range(len(self.df)), desc="Loading ECGs"):
            
            # --- CRITICAL FIX 2: Correctly mapping the resolution ---
            if self.sampling_rate == 500:
                rel_path = self.df.iloc[idx].filename_hr
            else:
                rel_path = self.df.iloc[idx].filename_lr 
                
            full_path = os.path.join(self.data_path, rel_path)
            
            # Read WFDB file
            record = wfdb.rdrecord(full_path)
            signal = record.p_signal 
            
            # Clean and convert
            signal = np.nan_to_num(signal)
            signal_tensor = torch.tensor(signal, dtype=torch.float32).transpose(0, 1)
            
            # Store permanently in RAM
            self.signals.append(signal_tensor)
        
    def _get_labels(self):
        def aggregate_diagnostic(y_dict):
            tmp = []
            for key in y_dict.keys():
                if key in self.diagnostic_df.index:
                    superclass = self.diagnostic_df.loc[key].diagnostic_class
                    if pd.notna(superclass):
                        tmp.append(superclass)
            return list(set(tmp))

        self.df['diagnostic_superclass'] = self.df.scp_codes.apply(aggregate_diagnostic)
        
        labels = np.zeros((len(self.df), len(self.superclasses)))
        for i, classes in enumerate(self.df['diagnostic_superclass']):
            for c in classes:
                if c in self.superclasses:
                    idx = self.superclasses.index(c)
                    labels[i, idx] = 1
        return torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Instantly fetches from RAM memory! Zero disk I/O.
        return self.signals[idx], self.labels[idx]

In [6]:
import torch
import torch.nn as nn
from mamba_ssm import Mamba

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm_x + self.eps)
        return self.weight * x_normed

class BiMambaBlock(nn.Module):
    """
    A Bidirectional Mamba block that processes the sequence forward and backward.
    Addition is used to merge the representations to keep dimensions stable.
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.forward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.backward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        
    def forward(self, x):
        # Apply normalization first (Pre-Norm architecture)
        normed_x = self.norm(x)
        
        # Forward pass
        fwd_out = self.forward_mamba(normed_x)
        
        # Backward pass (flip sequence along time dimension)
        bwd_out = self.backward_mamba(normed_x.flip(dims=[1])).flip(dims=[1])
        
        # Residual connection + merged state spaces
        return x + fwd_out + bwd_out

class AdvancedECGMamba(nn.Module):
    def __init__(self, in_channels=12, d_model=256, d_state=32, num_classes=5, num_layers=6):
        super().__init__()
        
        # 1. Strided Convolutional Stem (Patching/Downsampling)
        # Reduces sequence length from 5000 -> 1250, increasing receptive field
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, d_model // 2, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(d_model // 2),
            nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )
        
        # 2. Bidirectional Mamba Backbone
        self.layers = nn.ModuleList([
            BiMambaBlock(d_model=d_model, d_state=d_state) 
            for _ in range(num_layers)
        ])
        
        self.final_norm = RMSNorm(d_model)
        
        # 3. Classification Head
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        # x expected shape: (batch, channels=12, seq_len=5000)
        
        # Stem projection and downsampling
        x = self.stem(x) # Shape: (batch, d_model, 1250)
        
        # Transpose for Mamba: (batch, seq_len, d_model)
        x = x.transpose(1, 2)
        
        # Pass through Bi-Mamba blocks
        for layer in self.layers:
            x = layer(x)
            
        x = self.final_norm(x)
        
        # Global Max Pooling (often better than Average Pooling for sharp ECG peaks)
        x_max, _ = torch.max(x, dim=1)
        
        # Logits
        logits = self.head(x_max)
        return logits

In [7]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# 1. Define the dataloader
BASE_DIR = "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

train_dataset = PTBXLDataset(
    data_path=BASE_DIR,
    metadata_csv=os.path.join(BASE_DIR, "ptbxl_database.csv"),
    scp_csv=os.path.join(BASE_DIR, "scp_statements.csv"),
    sampling_rate=100,
    folds=[1, 2, 3, 4, 5, 6, 7, 8] # Explicitly isolated from Validation/Test
)

# --- THE FIX: num_workers=0 ---
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=0,        # --- CRITICAL FIX 3: Prevents RAM duplication crash ---
    pin_memory=True       # Speeds up RAM-to-GPU transfer
)

# 2. Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AdvancedECGMamba(
    in_channels=12,
    d_model=256,    
    d_state=32,     
    num_classes=5,  
    num_layers=6    
).to(device)

# 3. Advanced Optimization Setup
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()

# 4. The Training Loop
epochs = 30
max_norm = 1.0

print("Starting training with bfloat16 and RAM Caching...")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    
    for i, (signals, labels) in enumerate(train_dataloader):
        signals, labels = signals.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with autocast('cuda', dtype=torch.bfloat16):
            outputs = model(signals)
            loss = criterion(outputs, labels)
            
        if torch.isnan(loss):
            print(f"  [Warning] NaN loss at batch {i+1}, skipping gradient update.")
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        optimizer.step()
        
        total_loss += loss.item()
        valid_batches += 1
        
        if (i + 1) % 100 == 0:
            print(f"  Batch {i+1}/{len(train_dataloader)} - Current Loss: {loss.item():.4f}")
        
    scheduler.step()
    
    avg_loss = total_loss / max(valid_batches, 1)
    print(f"Epoch [{epoch+1}/{epochs}] | Avg Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

Filtered dataset to folds: [1, 2, 3, 4, 5, 6, 7, 8]. Total records to load: 17441
Pre-loading 100Hz dataset into RAM. This will take a moment...


Loading ECGs:   0%|          | 0/17441 [00:00<?, ?it/s]

Starting training with bfloat16 and RAM Caching...
  Batch 100/546 - Current Loss: 0.4298
  Batch 200/546 - Current Loss: 0.3560
  Batch 300/546 - Current Loss: 0.3641
  Batch 400/546 - Current Loss: 0.3478
  Batch 500/546 - Current Loss: 0.2844
Epoch [1/30] | Avg Loss: 0.3681 | LR: 0.000976
  Batch 100/546 - Current Loss: 0.3296
  Batch 200/546 - Current Loss: 0.3808
  Batch 300/546 - Current Loss: 0.3823
  Batch 400/546 - Current Loss: 0.3024
  Batch 500/546 - Current Loss: 0.3449
Epoch [2/30] | Avg Loss: 0.3099 | LR: 0.000905
  Batch 100/546 - Current Loss: 0.2941
  Batch 200/546 - Current Loss: 0.2719
  Batch 300/546 - Current Loss: 0.2560
  Batch 400/546 - Current Loss: 0.2295
  Batch 500/546 - Current Loss: 0.1427
Epoch [3/30] | Avg Loss: 0.2873 | LR: 0.000794
  Batch 100/546 - Current Loss: 0.2715
  Batch 200/546 - Current Loss: 0.2595
  Batch 300/546 - Current Loss: 0.3688
  Batch 400/546 - Current Loss: 0.1874
  Batch 500/546 - Current Loss: 0.2328
Epoch [4/30] | Avg Loss: 0.2

In [8]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score, hamming_loss, accuracy_score
from torch.amp import autocast

def evaluate_mamba_advanced(model, dataloader, device, threshold=0.5):
    """Evaluates the Mamba model with advanced multi-label metrics."""
    model.eval()
    
    all_probs = []
    all_targets = []
    
    print("Running advanced evaluation...")
    with torch.no_grad():
        for signals, labels in dataloader:
            signals = signals.to(device, non_blocking=True)
            
            with autocast('cuda', dtype=torch.bfloat16):
                logits = model(signals)
                probs = torch.sigmoid(logits) 
                
            all_probs.append(probs.cpu().float().numpy())
            all_targets.append(labels.numpy())
            
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    binary_preds = (all_probs >= threshold).astype(int)
    
    # --- CALCULATE METRICS ---
    try:
        macro_auc = roc_auc_score(all_targets, all_probs, average='macro')
        # AUPRC is critical for imbalanced medical datasets
        macro_auprc = average_precision_score(all_targets, all_probs, average='macro') 
    except ValueError:
        macro_auc, macro_auprc = float('nan'), float('nan')
        
    macro_f1 = f1_score(all_targets, binary_preds, average='macro', zero_division=0)
    
    # Exact Match Ratio (Did it get the exact combination of diseases right?)
    exact_match = accuracy_score(all_targets, binary_preds)
    
    # Hamming Loss (Fraction of incorrectly predicted individual labels)
    h_loss = hamming_loss(all_targets, binary_preds)
    
    # --- PRINT REPORT ---
    print("=" * 50)
    print("ADVANCED MULTI-LABEL METRICS")
    print("=" * 50)
    print(f"Macro AUROC:           {macro_auc:.4f}")
    print(f"Macro AUPRC:           {macro_auprc:.4f}  <-- Crucial for Imbalance")
    print(f"Macro F1-Score:        {macro_f1:.4f}")
    print(f"Exact Match Ratio:     {exact_match:.4f}  <-- Strict Subset Accuracy")
    print(f"Hamming Loss:          {h_loss:.4f}  <-- Lower is better")
    print("-" * 50)
    
    superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    print("\nPER-CLASS CLASSIFICATION REPORT:")
    print(classification_report(all_targets, binary_preds, target_names=superclasses, zero_division=0))
    
    return macro_auc, macro_auprc, macro_f1

In [10]:
# 1. Load the raw dataframe
df = pd.read_csv(os.path.join(BASE_DIR, "ptbxl_database.csv"), index_col='ecg_id')

# 2. Extract ONLY the test set (Fold 10)
test_df = df[df.strat_fold == 10].copy()

# 3. Save it to a temporary CSV so the Dataset class can read it
test_csv_path = "/kaggle/working/ptbxl_test.csv"
test_df.to_csv(test_csv_path)

# 4. Initialize the Test Dataset
test_dataset = PTBXLDataset(
    data_path=BASE_DIR,
    metadata_csv=test_csv_path, # Use the filtered CSV
    scp_csv=os.path.join(BASE_DIR, "scp_statements.csv"),
    sampling_rate=100
)

# 5. Initialize the Test Dataloader
test_dataloader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, # No need to shuffle test data
    num_workers=0, 
    pin_memory=True
)

# 6. Run the Evaluation
auc, f1 = evaluate_mamba_advanced(model, test_dataloader, device)

Pre-loading 100Hz dataset into RAM. This will take a moment...


Loading ECGs:   0%|          | 0/2203 [00:00<?, ?it/s]

Running advanced evaluation...
ADVANCED MULTI-LABEL METRICS
Macro AUROC:           0.9016
Macro AUPRC:           0.7465  <-- Crucial for Imbalance
Macro F1-Score:        0.7110
Exact Match Ratio:     0.6055  <-- Strict Subset Accuracy
Hamming Loss:          0.1243  <-- Lower is better
--------------------------------------------------

PER-CLASS CLASSIFICATION REPORT:
              precision    recall  f1-score   support

        NORM       0.79      0.91      0.85       964
          MI       0.74      0.66      0.70       553
        STTC       0.72      0.72      0.72       523
          CD       0.78      0.69      0.73       498
         HYP       0.69      0.46      0.55       263

   micro avg       0.76      0.74      0.75      2801
   macro avg       0.74      0.69      0.71      2801
weighted avg       0.76      0.74      0.75      2801
 samples avg       0.75      0.75      0.74      2801



ValueError: too many values to unpack (expected 2)